# 04 · Robustez: isolamento de falhas, *retry* e terminação

O laço dos notebooks anteriores assume que toda invocação de ferramenta tem sucesso. Em qualquer
cenário realista isso é falso: ferramentas acessam serviços externos que falham, recebem
argumentos malformados ou simplesmente não respondem. Na implementação ingênua, uma exceção
levantada por uma ferramenta **propaga-se e interrompe o agente**.

Este notebook introduz três salvaguardas, na ordem em que importam:

1. **Isolamento de falhas** — uma exceção em uma ferramenta é capturada e convertida em observação,
   permitindo que o modelo reaja a ela em vez de abortar a execução.
2. ***Retry*** — falhas transitórias (serviços instáveis) são reexecutadas algumas vezes antes de
   serem reportadas como definitivas.
3. **Terminação** — um limite de iterações impede que o laço prossiga indefinidamente caso o
   modelo nunca convirja para uma resposta.
4. **Conformidade com a plataforma** — o *deployment* dos modelos llama na NVIDIA aceita apenas
   *uma* invocação de ferramenta por turno; quando o modelo emite várias em paralelo, a requisição
   seguinte falha com erro 500. O laço descarta as invocações excedentes, mantendo a primeira, e o
   modelo reemite as demais nos turnos seguintes.

São, em essência, as garantias que frameworks de produção formalizam; aqui as implementamos
explicitamente para tornar visível o que está em jogo.


## Dependências

In [ ]:
!pip install -q openai

## Configuração do cliente

Empregamos a biblioteca `openai` apontando para o endpoint **gratuito da NVIDIA**
(`https://integrate.api.nvidia.com/v1`), que expõe uma interface compatível com a API da OpenAI.
A célula seguinte concentra os parâmetros de acesso: endpoint, modelo e credencial.

Para obter a credencial: criar conta em https://build.nvidia.com e gerar uma API key (prefixo
`nvapi-`). No Google Colab, a chave pode ser guardada uma única vez em *Secrets* (ícone de chave na
barra lateral) com o nome `NVIDIA_API_KEY`; a célula a seguir a detecta automaticamente. O modelo
padrão (`meta/llama-3.1-8b-instruct`) responde em cerca de um segundo por chamada e suporta
*function calling* de forma confiável.


In [ ]:
import os
import json
from getpass import getpass
from datetime import datetime
from openai import OpenAI

def _obter_api_key():
    try:                                  # 1) Colab Secrets (icone de chave a esquerda)
        from google.colab import userdata
        chave = userdata.get("NVIDIA_API_KEY")
        if chave:
            return chave
    except Exception:
        pass
    return os.environ.get("NVIDIA_API_KEY") or getpass("API key (nvapi-...): ")  # 2) env var ou 3) manual

BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL    = "meta/llama-3.1-8b-instruct"
API_KEY  = _obter_api_key()

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("Cliente configurado:", BASE_URL, "| modelo:", MODEL)


## Uma ferramenta sujeita a falhas

Para tornar o problema concreto e determinístico, modelamos um serviço instável: `cotacao_dolar`
falha na primeira tentativa e tem sucesso na seguinte. Esse padrão — falha transitória seguida de
recuperação — é exatamente o que justifica uma política de *retry*.


In [ ]:
_tentativas_cotacao = 0

def cotacao_dolar() -> str:
    # Servico instavel: falha em tentativas impares, sucede nas pares
    global _tentativas_cotacao
    _tentativas_cotacao += 1
    if _tentativas_cotacao % 2 == 1:
        raise RuntimeError("servico de cotacao temporariamente indisponivel")
    return "R$ 5,43"


def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Erro: {e}"


TOOL_FUNCTIONS = {"cotacao_dolar": cotacao_dolar, "calculate": calculate}

TOOLS = [
    {"type":"function","function":{
        "name":"cotacao_dolar",
        "description":"Retorna a cotacao atual do dolar em reais.",
        "parameters":{"type":"object","properties":{},"required":[]}}},
    {"type":"function","function":{
        "name":"calculate",
        "description":"Avalia uma expressao matematica.",
        "parameters":{"type":"object",
            "properties":{"expression":{"type":"string","description":"ex.: '5.43 * 100'"}},
            "required":["expression"]}}},
]


## O laço ingênuo: uma falha o interrompe

A versão abaixo executa a ferramenta sem proteção. Quando `cotacao_dolar` falha na primeira
tentativa, a exceção propaga-se e o agente é interrompido. Envolvemos a chamada em `try/except`
**na célula de demonstração** apenas para que o erro seja exibido em vez de abortar o kernel — o
ponto é que, sem tratamento dentro do laço, não há recuperação possível.


In [ ]:
def run_agent_ingenuo(task, model=MODEL):
    messages = [
        {"role":"system","content":"Voce e um assistente util. Use ferramentas quando precisar."},
        {"role":"user","content":task},
    ]
    while True:
        r = client.chat.completions.create(model=model, messages=messages, tools=TOOLS, tool_choice="auto")
        msg = r.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = TOOL_FUNCTIONS[tc.function.name](**args)   # sem protecao
            messages.append({"role":"tool","tool_call_id":tc.id,"content":str(result)})


In [ ]:
_tentativas_cotacao = 0  # reinicia o servico instavel para a demonstracao
try:
    print(run_agent_ingenuo("Qual a cotacao atual do dolar?"))
except Exception as e:
    print("AGENTE INTERROMPIDO:", type(e).__name__, "-", e)


## O laço robusto

A versão robusta difere em três pontos, todos internos ao laço:

- a execução de cada ferramenta é envolvida em `try/except`, com *retry* até `max_retries`;
- exceções persistentes e argumentos malformados (JSON inválido) tornam-se observações textuais
  devolvidas ao modelo, não falhas fatais;
- o laço externo é limitado a `max_iter` iterações.

Observe, na saída, a primeira tentativa de `cotacao_dolar` falhando e a segunda sendo bem-sucedida,
sem que o agente seja interrompido.


In [ ]:
def run_agent(task, model=MODEL, max_iter=10, max_retries=3, verbose=True):
    messages = [
        {"role":"system","content":"Voce e um assistente util. Use ferramentas quando precisar. Quando tiver a resposta final, responda sem chamar ferramentas."},
        {"role":"user","content":task},
    ]
    for _ in range(max_iter):
        r = client.chat.completions.create(model=model, messages=messages, tools=TOOLS, tool_choice="auto")
        msg = r.choices[0].message
        if msg.tool_calls and len(msg.tool_calls) > 1:
            msg.tool_calls = msg.tool_calls[:1]   # endpoint aceita 1 tool call por turno
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                messages.append({"role":"tool","tool_call_id":tc.id,
                                 "content":"Erro: argumentos invalidos (JSON malformado)."})
                continue

            fn = TOOL_FUNCTIONS.get(name)
            if fn is None:
                resultado = f"Ferramenta desconhecida: {name}"
            else:
                resultado = None
                for tentativa in range(1, max_retries + 1):
                    try:
                        resultado = str(fn(**args))
                        break
                    except Exception as e:
                        if verbose:
                            print(f"   [{name}] tentativa {tentativa}/{max_retries} falhou: {e}")
                        if tentativa == max_retries:
                            resultado = f"Erro apos {max_retries} tentativas: {e}"
            messages.append({"role":"tool","tool_call_id":tc.id,"content":resultado})

    return "[limite de iteracoes atingido sem resposta final]"


In [ ]:
_tentativas_cotacao = 0  # reinicia o servico instavel
print(run_agent("Qual a cotacao atual do dolar?"))


## Exercício

Modifique `cotacao_dolar` para falhar **sempre** (remova a condição de sucesso) e execute novamente
o agente robusto. Observe que ele não é interrompido: após `max_retries` tentativas, o erro é
devolvido ao modelo como observação, e o agente ainda produz uma resposta final — degradação
graciosa em vez de falha catastrófica. Em seguida, reduza `max_iter` para `1` e proponha uma tarefa
que exija duas ferramentas, observando o efeito do limite de terminação.


---
Continuação: `05_delegacao.ipynb` — orquestração com um modelo pequeno que delega o raciocínio
mais custoso a um modelo maior, exposto como ferramenta.
